<p style="text-align:center"><img src="../images/04_00_spatial_stat.png" alt="Spatial Statistics" width="100%"></p>


# Module Map, Architecture, and Learning Paths

This notebook is Foundation Chapter 0.3 of 3. It maps Module 04 end to end: the `geolattice.spatial_statistics` package layout, every Part and notebook, a reproducible setup cell, and role-based learning paths.

| Chapter | Notebook | Focus |
|---|---|---|
| **0.1** | [`04_00_01_spatial_statistics_introduction.ipynb`](04_00_01_spatial_statistics_introduction.ipynb) | What spatial statistics is and why it matters |
| **0.2** | [`04_00_02_core_theoretical_concepts.ipynb`](04_00_02_core_theoretical_concepts.ipynb) | Tobler, data types, stationarity, MAUP, CSR, weights |
| **0.3** (this) | [`04_00_03_module_map_and_learning_paths.ipynb`](04_00_03_module_map_and_learning_paths.ipynb) | Architecture, Part overview, setup, learning paths |


## Overview

| Section | What you get |
|---|---|
| Package architecture | Tree of `geolattice.spatial_statistics` submodules aligned to Parts 1–9 |
| Module overview | Dataset, library entry point, and notebook table for every Part |
| Setup | Root discovery, imports, style, version check |
| Learning paths | Full foundations, epidemiology, environment, urban/policy, and a dependency map |

API philosophy shared by all submodules:

- **Input:** GeoDataFrame or numpy/pandas arrays with CRS metadata
- **Output:** result objects with `.summary()`, `.plot()`, and `.export()` methods
- **Reproducibility:** randomisation / permutation tests accept a `seed` argument


## API overview (`spatial_statistics`)

```
- weights
- pointpattern
- esda
- regression
- cluster
- bayes
- gwmodel
- interpolate
- sampling
```


## Setup

Locate the repository root and import the spatial-statistics submodules.

**Installation.** This notebook needs installation option 3 (core, spatial analysis and spatial statistics): `pip install "geolattice[statistics]"`. See "Installation options" in the GeoLattice README.

In [1]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").is_file() and (candidate / "Data").is_dir():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find the GeoLattice repository root.")
sys.path.insert(0, str(ROOT))

import geolattice
import geolattice.spatial_statistics as sstat
from geolattice.spatial_statistics import (
    bayes,
    cluster,
    esda,
    gwmodel,
    interpolate,
    pointpattern,
    regression,
    sampling,
    weights,
)
from geolattice.viz import apply_map_style

apply_map_style()

print(f"geolattice: {geolattice.__version__}")
print(f"geolattice.spatial_statistics: {sstat.__version__}")
print("Imported:", ", ".join([
    "weights", "pointpattern", "esda", "regression", "cluster",
    "bayes", "gwmodel", "interpolate", "sampling",
]))


geolattice: 0.3.49
geolattice.spatial_statistics: 0.3.49
Imported: weights, pointpattern, esda, regression, cluster, bayes, gwmodel, interpolate, sampling


## GeoLattice Spatial Statistics Architecture

Module 04 is built around **GeoLattice** (`geolattice.spatial_statistics`), a purpose-built Python library that wraps and extends the PyData spatial ecosystem.

```
geolattice.spatial_statistics
├── pointpattern   ← Part 1: events, windows, intensity, K/L/G/F/J functions
├── weights        ← Parts 2–4: spatial weights matrices, row-standardisation
├── esda           ← Part 2: Moran's I, Geary's C, LISA (local Moran, Getis-Ord)
├── regression     ← Part 3: OLS, SLM, SEM, SDM, GNS, SARAR, GWR helpers
├── cluster        ← Part 4: Getis-Ord Gi*, DBSCAN, SaTScan, SKATER, REDCAP
├── bayes          ← Part 5: EB smoothing, BYM/CAR models, MCMC
├── gwmodel        ← Part 6: GWSS, GWR, MGWR, GW-PCA, spatio-temporal GWR
├── interpolate    ← Part 7: IDW, RBF, ordinary/universal kriging, areal interp
├── sampling       ← Part 8: SRS, systematic, stratified, BAS, GRTS designs
└── spacetime      ← Part 9: (stub: uses esda, cluster, gwmodel directly)
```

Use this tree as a lookup: if you know the scientific question from Chapter 0.1, the matching submodule is the import you need.


## Module Overview

### Point Pattern Analysis

> **What it is:** A set of event locations in a bounded study window; the *locations themselves* carry the information.  
> **Core question:** Are events clustered, dispersed, or consistent with Complete Spatial Randomness?  
> **Dataset:** Buffalo crime incidents 2018 (`crime_data_2018.csv`, `buffalo_city.shp`)  
> **Library:** `geolattice.spatial_statistics.pointpattern`

| Notebook | Focus |
|---|---|
| **04.01.1** Point Pattern Fundamentals | Point processes, observation windows, marks, and CSR as the null hypothesis |
| **04.01.2** Spatial Intensity | First-order properties: raw intensity, KDE, bandwidth selection, covariate-adjusted intensity |
| **04.01.3** Second-Order Functions | $K(r)$, $L(r)$, $G(r)$, $F(r)$, $J(r)$: theory, estimation, clustered vs. regular patterns |
| **04.01.4** Hypothesis Testing | Monte-Carlo envelopes, global and pointwise tests, multiple-testing corrections, power |
| **04.01.5** Areal Integration & Pitfalls | Aggregating points to areal counts, edge effects, inhomogeneous extensions, MAUP |

**Key theory.** A **spatial point process** $\Phi$ is a random, locally finite set of points in $\mathbb{R}^2$. The intensity $\lambda(s)$ is the expected number of events per unit area near $s$. Under CSR, $\lambda(s) = \lambda$ (constant). The second-order intensity underpins Ripley's $K$-function:

$$K(r) = \frac{1}{\lambda}\,\mathbb{E}\bigl[\text{events within distance }r\text{ of a typical event}\bigr]$$

Under CSR, $K(r) = \pi r^2$. Values above signal clustering; values below signal regularity.

### Spatial Autocorrelation

> **What it is:** Lattice/areal data—one numeric value per polygon.  
> **Core question:** Is the spatial arrangement of values more similar (or dissimilar) than chance?  
> **Dataset:** US county diabetes prevalence, Northeast and Atlantic regions  
> **Library:** `geolattice.spatial_statistics.weights`, `esda`, `viz`

| Notebook | Focus |
|---|---|
| **04.02.1** Concepts, Data & Foundations | Tobler's first law, dependence vs. heterogeneity, MAUP, dataset introduction |
| **04.02.2** Spatial Weights | Contiguity (rook, queen), KNN, distance-band, row-standardisation, higher-order weights |
| **04.02.3** Visualization & Exploratory Analysis | Choropleths, Moran scatterplot, spatial lag plots, quantile maps |
| **04.02.4** Global Autocorrelation | Moran's $I$, Geary's $C$, Getis-Ord $G$: permutation inference |
| **04.02.5** Local Autocorrelation: LISA | Local Moran's $I_i$, Getis-Ord $G_i^*$, LISA cluster maps |
| **04.02.6** Sensitivity, Pitfalls & Exercises | Weight-choice sensitivity, boundary effects, MAUP revisited |

**Key theory.** Moran's $I$:

$$I = \frac{n}{S_0}\cdot\frac{\sum_i\sum_j w_{ij}(z_i-\bar{z})(z_j-\bar{z})}{\sum_i(z_i-\bar{z})^2},\qquad S_0=\sum_i\sum_j w_{ij}$$

Under spatial randomness, $\mathbb{E}[I] = -1/(n-1)\approx 0$. Local Moran $I_i$ identifies HH, LL, HL, and LH clusters.

### Spatial Regression

> **What it is:** Regression that accounts for spatial autocorrelation in residuals or in the response.  
> **Core question:** What covariates drive a spatial outcome, and how do we obtain valid inference despite dependence?  
> **Dataset:** `diabetes_atlantic.shp` (666 counties)  
> **Library:** `geolattice.spatial_statistics.regression`

| Notebook | Focus |
|---|---|
| **04.03.1** Foundations | Why OLS fails; substantive vs. nuisance dependence; model taxonomy |
| **04.03.2** OLS Baseline & Diagnostics | Moran $I$ on residuals; LM-lag, LM-error, robust variants |
| **04.03.3** Model Selection | LM decision tree; AIC/BIC; cross-validation |
| **04.03.4** Model Families | SLM, SEM, SDM, SARAR, GNS; ML and GMM estimation |
| **04.03.5** Comparison & Reporting | Marginal effects (direct/indirect/total); spatial multiplier |
| **04.03.6** Limits & Connections | Endogeneity, panels, connections to GWR |
| **04.03.7** Exercises | Structured exercises on the Atlantic dataset |

**Key theory.** Spatial Lag (SLM/SAR): $y = \rho W y + X\beta + \varepsilon$. Spatial Error (SEM): $y = X\beta + u$, $u = \lambda W u + \varepsilon$. SAR models substantive interaction; SEM models nuisance dependence. Lagrange Multiplier tests on OLS residuals guide the choice.

### Spatial Cluster Analysis

> **What it is:** Methods that locate statistically significant concentrations of high (or low) values.  
> **Core question:** Where are the hotspots, and are they statistically significant?  
> **Dataset:** `diabetes_northeast.shp` (217 counties)  
> **Library:** `geolattice.spatial_statistics.cluster`

| Notebook | Focus |
|---|---|
| **04.04.1** Concepts & Overview | Value clusters, location clusters, density clusters, spatial outliers |
| **04.04.2** Hotspot Detection | Getis-Ord $G_i^*$, local Moran, FDR corrections |
| **04.04.3** Density & Location Clustering | DBSCAN, HDBSCAN, OPTICS on points and areal centroids |
| **04.04.4** Scan Statistic | Kulldorff spatial scan (Poisson, Bernoulli); circular and flexible windows |
| **04.04.5** Regionalisation | SKATER, REDCAP, WARD-spatial |
| **04.04.6** Comparing & Validating | Silhouette, Calinski–Harabász, spatial compactness |
| **04.04.7** Bridges | Hand-offs to regression (Part 3) and disease mapping (Part 5) |
| **04.04.8** Exercises | Crime hotspots, epidemic clusters, zoning |

**Key theory.** Kulldorff's scan maximises a likelihood-ratio statistic over windows $A$ and obtains $p$-values by Monte Carlo under the null—no asymptotic tables required.

### Disease Mapping & Bayesian Smoothing

> **What it is:** Small-area estimation that produces stable, uncertainty-aware disease rate maps.  
> **Core question:** How do we distinguish true spatial patterns from sampling noise in small-population areas?  
> **Dataset:** `diabetes_northeast.shp` (217 counties)  
> **Library:** `geolattice.spatial_statistics.bayes`

| Notebook | Focus |
|---|---|
| **04.05.1**–**04.05.4** | Objectives, small-area problem, data prep, raw SMR estimates |
| **04.05.5** | Empirical Bayes (global Marshall; local Clayton–Kaldor) |
| **04.05.6**–**04.05.7** | BYM / ICAR models; MCMC (PyMC / NumPyro) |
| **04.05.8**–**04.05.9** | DIC/WAIC/LOO; shrinkage and exceedance maps |
| **04.05.10**–**04.05.11** | Reporting & ethics; handoffs and exercises |

**Key theory.** BYM: $y_i \sim \operatorname{Poisson}(E_i\exp(\alpha+\phi_i+\theta_i))$ with $\phi\sim\operatorname{ICAR}(W,\sigma_\phi^2)$ (structured) and $\theta_i\sim\mathcal{N}(0,\sigma_\theta^2)$ (unstructured).

### Geographically Weighted Models

> **What it is:** Locally calibrated regression and summaries whose coefficients vary in space.  
> **Core question:** Do relationships between variables change across the study region?  
> **Dataset:** `diabetes_atlantic.shp` (666 counties)  
> **Library:** `geolattice.spatial_statistics.gwmodel`

| Notebook | Focus |
|---|---|
| **04.06.1**–**04.06.3** | Motivation, data, kernels & bandwidth selection |
| **04.06.4**–**04.06.5** | GWSS exploratory summaries; GWR inference and local $t$ |
| **04.06.6**–**04.06.8** | MGWR, GW-PCA; diagnostics; comparison with OLS/SAR |
| **04.06.9**–**04.06.11** | Spatial ML ensembles; GTWR; geographic deep models |
| **04.06.12**–**04.06.14** | Scaling; troubleshooting; exercises |

**Key theory.** $\hat{\beta}(s_i)=(X^\top W(s_i)X)^{-1}X^\top W(s_i)y$ with kernel weights decaying from $s_i$. Bandwidth $b$ trades local fidelity against noise; MGWR allows a separate $b_k$ per covariate.

### Spatial Interpolation

> **What it is:** Prediction of a continuous field at unsampled locations.  
> **Core question:** What is the value at an unmeasured location, and how uncertain is that estimate?  
> **Dataset:** `CA_pm25_2025.csv` (163 California monitors)  
> **Library:** `geolattice.spatial_statistics.interpolate`

| Notebook | Focus |
|---|---|
| **04.07.1**–**04.07.2** | Concepts; exploratory variogram and trend detection |
| **04.07.3** | Deterministic methods: IDW, RBF, natural neighbour |
| **04.07.4**–**04.07.5** | Variogram modelling; ordinary/universal kriging; validation |
| **04.07.6**–**04.07.7** | Regression kriging, co-kriging; areal / dasymetric interpolation |
| **04.07.8** | Exercises: PM2.5 reconstruction |

**Key theory.** Ordinary kriging is a BLUP: $\hat{Z}(s_0)=\sum_i\lambda_i Z(s_i)$ with $\sum_i\lambda_i=1$, weights from the variogram system.

### Spatial Sampling & Design

> **What it is:** Selecting spatial samples that support valid, efficient inference.  
> **Core question:** Where should we place observations to minimise cost and maximise precision?  
> **Dataset:** `diabetes_usa.csv` (3,107 US counties as a known population laboratory)  
> **Library:** `geolattice.spatial_statistics.sampling`

| Notebook | Focus |
|---|---|
| **04.08.1** | Design-based vs. model-based inference; spatial balance |
| **04.08.2** | SRS, systematic, stratified, cluster, BAS, GRTS |
| **04.08.3** | Horvitz–Thompson estimation and design effect |
| **04.08.4** | Monitoring network design: add/drop optimisations |
| **04.08.5** | Exercises |

**Key theory.** $\hat{T}_{HT}=\sum_{i\in S} y_i/\pi_i$ is unbiased under any probability sample. Spatial balance (GRTS, BAS) reduces variance by exploiting positive autocorrelation.

### Spatio-Temporal Analysis

> **What it is:** Spatial methods extended to multiple time points—the space–time cube.  
> **Core question:** How do spatial patterns, clusters, and relationships evolve over time?  
> **Dataset:** County panel 1998–2012; Buffalo crime 2018  
> **Library:** `geolattice.spatial_statistics.spacetime` (builds on `esda`, `cluster`, `gwmodel`)

| Notebook | Focus |
|---|---|
| **04.09.1** | Space–time cube; separable vs. non-separable structure |
| **04.09.2** | Space–time weights; space–time Moran $I$ |
| **04.09.3** | Emerging hotspot analysis (Mann–Kendall + Getis-Ord) |
| **04.09.4** | Space–time KDE; Knox test; STDBSCAN |
| **04.09.5** | Space–time kriging; product-sum covariance |
| **04.09.6** | Spatial panels; GTWR; temporal residual autocorrelation |

**Key theory.** Separability means $\lambda(s,t)=\lambda_s(s)\cdot\lambda_t(t)$. The Knox statistic counts pairs that are close in *both* space and time and compares against time-permuted nulls.


## Suggested Learning Paths

The nine Parts are modular. Pick a path based on your goal.

### Path A — Complete Foundations (all levels)

```
04.00 → 04.01 → 04.02 → 04.03 → 04.04 → 04.05 → 04.06 → 04.07 → 04.08 → 04.09
```

Parts 1 → 2 → 3 are the natural exploratory → confirmatory progression.

### Path B — Epidemiology / Public Health

Disease mapping, cluster detection, and the Bayesian toolchain.

```
04.00 → 04.02 → 04.04 → 04.05 → 04.03 → 04.09
```

### Path C — Environmental / Geostatistics

Field data, interpolation, and sampling design for monitoring networks.

```
04.00 → 04.01 → 04.07 → 04.08 → 04.09
```

### Path D — Urban / Economic / Policy

Regression, local heterogeneity, and cluster analysis for areal data.

```
04.00 → 04.02 → 04.03 → 04.04 → 04.06 → 04.09
```

### Path E — Quick Reference (advanced users)

Jump directly to the method you need using the Part tables in Section 2.

### Dependency map

```
               [04.01 Point Pattern]
                        │
                        ▼
  [04.02 Autocorrelation] ───► [04.03 Regression] ───► [04.06 GWR]
          │                           │                      │
          ▼                           ▼                      │
  [04.04 Clustering]         [04.05 Disease Mapping] ◄───────┘
          │
          ▼
  [04.07 Interpolation] ───► [04.08 Sampling]
          │
          ▼
  [04.09 Spatio-Temporal]  (uses all of the above)
```

---

**Ready to start?** Foundation Chapters 0.1–0.3 are complete. Proceed to Part 1 — Point Pattern Analysis — or follow your chosen path above. If you skimmed Chapters 0.1–0.2, keep [`04_00_02_core_theoretical_concepts.ipynb`](04_00_02_core_theoretical_concepts.ipynb) open as a glossary while you work.
